In [1]:
TITLE = "The indispensable Calvin and Hobbes"

In [2]:
import os

from dotenv import load_dotenv

load_dotenv()

True

# Google Books API

In [3]:
import httpx
from langchain_core.tools import tool

GOOGLE_BOOKS_API_URL = "https://www.googleapis.com/books/v1/volumes"

# Google Books returns ISO 639-1 codes (e.g. "en"), but your BookInfo
# schema wants full names ("English") — map the common ones here.
LANGUAGE_MAP = {
    "en": "English",
    "fr": "French",
    "de": "German",
    "es": "Spanish",
    "it": "Italian",
    "pt": "Portuguese",
    "nl": "Dutch",
    "ru": "Russian",
    "zh": "Chinese",
    "ja": "Japanese",
    "ko": "Korean",
    "ar": "Arabic",
    "hi": "Hindi",
    "bn": "Bengali",
}


def _normalize_date(date_str: str) -> str:
    """Google Books' publishedDate can be 'YYYY', 'YYYY-MM', or
    'YYYY-MM-DD'. Pad missing parts so it's a valid date string."""
    if not date_str:
        return "Unknown"
    parts = date_str.split("-")
    year = parts[0]
    month = parts[1] if len(parts) > 1 else "01"
    day = parts[2] if len(parts) > 2 else "01"
    return f"{year}-{month}-{day}"


@tool
def search_book(title: str) -> str:
    """Search for a book by title using the Google Books API and return
    metadata (title, authors, publisher, page count, language, published
    date) for the top matching results."""
    params = {"q": f"intitle:{title}", "maxResults": 5}
    api_key = os.getenv("GOOGLE_BOOKS_API_KEY")
    if api_key:
        params["key"] = api_key


    try:
        response = httpx.get(GOOGLE_BOOKS_API_URL, params=params, timeout=10.0)
        response.raise_for_status()
        data = response.json()
    except httpx.HTTPError as e:
        return f"Error calling Google Books API: {e}"

    items = data.get("items")
    if not items:
        return f"No books found matching the title '{title}'."

    results = []
    for item in items:
        vi = item.get("volumeInfo", {})
        lang_code = vi.get("language", "")
        results.append(
            {
                "title": vi.get("title", "Unknown"),
                "authors": vi.get("authors", ["Unknown"]),
                "publisher": vi.get("publisher", "Unknown"),
                "page_count": vi.get("pageCount"),
                "language": LANGUAGE_MAP.get(lang_code, lang_code or "Unknown"),
                "published_date": _normalize_date(vi.get("publishedDate", "")),
            }
        )

    return "\n\n".join(
        f"Result {i + 1}:\n"
        f"  Title: {r['title']}\n"
        f"  Author(s): {', '.join(r['authors'])}\n"
        f"  Publisher: {r['publisher']}\n"
        f"  Page count: {r['page_count']}\n"
        f"  Language: {r['language']}\n"
        f"  Published date: {r['published_date']}"
        for i, r in enumerate(results)
    )

In [4]:
res = search_book.invoke({"title": TITLE})
print(res)

Result 1:
  Title: The Indispensable Calvin and Hobbes
  Author(s): Bill Watterson
  Publisher: Andrews McMeel Publishing
  Page count: 262
  Language: English
  Published date: 1992-06-01

Result 2:
  Title: The Essential Calvin And Hobbes
  Author(s): Bill Watterson
  Publisher: Andrews McMeel Publishing
  Page count: 262
  Language: English
  Published date: 1988-01-01

Result 3:
  Title: The Essential Calvin and Hobbes
  Author(s): Bill Watterson
  Publisher: Unknown
  Page count: 254
  Language: English
  Published date: 1988-01-01

Result 4:
  Title: The Essential Calvin and Hobbes
  Author(s): Bill Watterson
  Publisher: Turtleback Books
  Page count: 254
  Language: English
  Published date: 1988-01-01

Result 5:
  Title: The Complete Calvin and Hobbes
  Author(s): Bill Watterson
  Publisher: Andrews McMeel Publishing
  Page count: 492
  Language: English
  Published date: 2005-09-01


# OPEN LIBRARY SEARCH API

In [5]:
OPEN_LIBRARY_SEARCH_URL = "https://openlibrary.org/search.json"

# Open Library uses ISO 639-2/B three-letter codes (e.g. "eng"), unlike
# Google Books' two-letter codes above — needs its own map.
OL_LANGUAGE_MAP = {
    "eng": "English",
    "fre": "French",
    "ger": "German",
    "spa": "Spanish",
    "ita": "Italian",
    "por": "Portuguese",
    "dut": "Dutch",
    "rus": "Russian",
    "chi": "Chinese",
    "jpn": "Japanese",
    "kor": "Korean",
    "ara": "Arabic",
    "hin": "Hindi",
    "ben": "Bengali",
}


@tool
def search_book_openlibrary(title: str) -> str:
    """Fallback book search using the Open Library API. Note:
    published_date here is year-only (Jan 1) — the search endpoint
    doesn't expose a full publication date."""
    params = {
        "title": title,
        "limit": 5,
        "fields": "title,author_name,publisher,number_of_pages_median,language,first_publish_year",
    }

    try:
        response = httpx.get(OPEN_LIBRARY_SEARCH_URL, params=params, timeout=10.0)
        response.raise_for_status()
        data = response.json()
    except httpx.HTTPError as e:
        return f"Error calling Open Library API: {e}"

    docs = data.get("docs")
    if not docs:
        return f"No books found matching the title '{title}' on Open Library."

    results = []
    for doc in docs:
        lang_codes = doc.get("language", [])
        lang = OL_LANGUAGE_MAP.get(lang_codes[0], lang_codes[0]) if lang_codes else "Unknown"
        publishers = doc.get("publisher", ["Unknown"])
        year = doc.get("first_publish_year")
        results.append({
            "title": doc.get("title", "Unknown"),
            "authors": doc.get("author_name", ["Unknown"]),
            "publisher": publishers[0] if publishers else "Unknown",
            "page_count": doc.get("number_of_pages_median"),
            "language": lang,
            "published_date": f"{year}-01-01" if year else "Unknown",
        })

    return "\n\n".join(
        f"Result {i+1}:\n"
        f"  Title: {r['title']}\n"
        f"  Author(s): {', '.join(r['authors'])}\n"
        f"  Publisher: {r['publisher']}\n"
        f"  Page count: {r['page_count'] if r['page_count'] else 'Not available'}\n"
        f"  Language: {r['language']}\n"
        f"  Published date: {r['published_date']} (year only)"
        for i, r in enumerate(results)
    )

In [6]:
res = search_book_openlibrary.invoke({"title": TITLE})
print(res)

Result 1:
  Title: The Indispensable Calvin and Hobbes
  Author(s): Bill Watterson
  Publisher: Turtleback Books
  Page count: 255
  Language: English
  Published date: 1992-01-01 (year only)


# Tavily WEB SEARCH TOOL

In [7]:
from langchain.tools import tool
from tavily import TavilyClient

tavily_client = TavilyClient()


@tool
def search_book_tavily(title: str) -> str:
    """
    Last-resort fallback book search using Tavily's web search API. Use
    this only after search_book (Google Books) and search_book_openlibrary
    (Open Library) have failed or left gaps in publisher, page_count, or
    language. Results are unstructured web snippets, not typed book
    metadata — extract fields carefully and only when confident.
    """

    try:
        data = tavily_client.search(
            query=f"{title} book author publisher page count language original publication date"
        )
    except Exception as e:
        return f"Error calling Tavily API: {e}"

    results = data.get("results", [])
    if not results and not data.get("answer"):
        return f"No web results found for '{title}'."

    lines = []
    if data.get("answer"):
        lines.append(f"Summarized answer: {data['answer']}")

    for i, r in enumerate(results, start=1):
        lines.append(
            f"\nResult {i}: {r.get('title', 'Untitled')} ({r.get('url', '')})\n"
            f"{r.get('content', '')[:500]}"
        )

    return "\n".join(lines)

In [8]:
res = search_book_tavily.invoke({"title": TITLE})
print(res)


Result 1: The indispensable Calvin and Hobbes (https://www.readinglength.com/work/WiHvhML)
Enter your reading speed

You can take one of our WPM reading speed tests to find your reading speed.

Create a free account to track your reading progress, build your reading list, and set reading goals.

Get it on AmazonBuy on Bookshop.org

We earn a commission on purchases

### Author

 Bill Watterson

### Publication

1992 - Andrews and McMeel, Kansas City, Missouri

### Language

English

### Word Count

63,750 words, Guess

### Page Count

255 pages

### Identifiers [...] ### Identifiers



Result 2: The Essential Calvin and Hobbes - Andrews McMeel Publishing (https://publishing.andrewsmcmeel.com/book/the-essential-calvin-and-hobbes-a-calvin-and-hobbes-treasury)
A mix of classic black-and-white daily strips and vividly colored Sunday comics showcases Bill Watterson’s incredible artistic range and sharp, timeless humor. With over 250 pages, this collection captures the essence of childhood 

# SYSTEM PROMPT

In [9]:
SYSTEM_PROMPT = """You are a Book Information Retrieval Agent. Your only job is to find accurate, factual information about a book given its title and return it in a structured format.

## Task
The user will give you a book title. Your job:
1. Identify the correct book — if multiple editions or books share a similar title, if it's genuinely ambiguous and no tool result resolves it, pick the most well-known / most recent edition and note the assumption in your reasoning (not in the final structured output).
2. Use your available tools to search for and verify the book's details. Never rely on memory alone for facts like page count or publisher — always confirm with a tool call when a search tool is available.

## Rules
- Do not fabricate or guess values. If a tool returns no result for the given title, say so instead of inventing plausible-sounding data.
- If a field's exact value can't be found (e.g. page count varies by edition), use the most commonly cited/paperback edition value and be consistent — don't mix data from two different editions for the same book.
- If the title doesn't match any known book, respond that no matching book was found rather than returning a guessed BookInfo object.
- Do not add commentary, opinions, or a review of the book — only the requested factual fields.
- If the user's title is misspelled or partial, use your best judgment to match it to the intended book and proceed.

## Tools
You have access to three book search tools, to be tried in this order:
1. search_book — searches Google Books. Always try this first.
2. search_book_openlibrary — fallback search on Open Library. Use it only
   if search_book returns no results, or its results are missing
   page_count or language.
3. search_book_tavily — general web search fallback, built for LLM
   consumption. Use this only as a last resort, after both book-specific
   tools have failed or left gaps — e.g. for obscure, non-English, or
   very recent titles not yet indexed by the book APIs. Because this
   returns web snippets rather than structured metadata, only extract
   page_count/publisher/language values you're confident are accurate;
   never guess from ambiguous text.

Don't call all three tools by default — only escalate to the next one
when the previous tool leaves a genuine gap."""

# Main Agent

In [10]:
from datetime import date

from langchain.agents import create_agent
from pydantic import BaseModel


class BookInfo(BaseModel):
    title: str
    author: str
    publisher: str
    page_count: int
    language: str
    published_date: date


agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[search_book, search_book_openlibrary, search_book_tavily],
    system_prompt=SYSTEM_PROMPT,
    response_format=BookInfo,
)

In [11]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": TITLE}]}
)

In [12]:
# from langchain.messages import HumanMessage

# message = HumanMessage(content="What's the weather in San Francisco?")
# result = agent.invoke(
#     {"messages": [message]}
# )
print(result["messages"][-1].content_blocks)

[{'type': 'text', 'text': '{\n  "title": "The Indispensable Calvin and Hobbes",\n  "author": "Bill Watterson",\n  "publisher": "Andrews McMeel Publishing",\n  "page_count": 262,\n  "language": "English",\n  "published_date": "1992-06-01"\n}', 'extras': {'signature': 'El4KXAFpFH0T10+W1J0oKVPbNMw/4DxSBAo0NtLOZBNWH63xZqMIiIFTjb/pz9BzLjOvUh8B4dwwMRenfS63q0GYfJ7nqyyarw5/F2Tq85/puB3KUbnWPAPxc/tJelHu'}}]


In [13]:
print(result["messages"][-1].content_blocks[0]["text"])

{
  "title": "The Indispensable Calvin and Hobbes",
  "author": "Bill Watterson",
  "publisher": "Andrews McMeel Publishing",
  "page_count": 262,
  "language": "English",
  "published_date": "1992-06-01"
}


In [14]:
import json

print(json.loads(result["messages"][-1].content_blocks[0]["text"])["title"])

The Indispensable Calvin and Hobbes
